# Part 3: Statistical & Machine Learning Analysis
### Socioeconomic Distress and Violent Crime

*Requires `processed_data.csv` from Part 1.*

Covers baseline OLS, nonlinear OLS specifications, Random Forest and XGBoost models, SHAP explainability, and poverty/unemployment threshold detection.

## Setup — Load Processed Data & Packages

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import shap
from IPython.display import Image, display

os.makedirs('output', exist_ok=True)
analysis_df = pd.read_csv('processed_data.csv')

OUTCOME = 'crime_rate_per_100000'
UNEM = 'Unemployment_rate_2023'
POV = 'PCTPOVALL_2023'
CONTROLS = [
    'MEDHHINC_2023',
    "Percent of adults who are not high school graduates, 2019-23",
    "Percent of adults with a bachelor's degree or higher, 2019-23",
]

FEATURES = []
for col in [UNEM, POV] + CONTROLS:
    if col in analysis_df.columns:
        FEATURES.append(col)

SHORT = {
    OUTCOME: 'Crime Rate per 100k',
    UNEM: 'Unemployment Rate 2023',
    POV: 'Poverty Rate 2023',
    'MEDHHINC_2023': 'Median HH Income',
    "Percent of adults who are not high school graduates, 2019-23": 'No HS Diploma %',
    "Percent of adults with a bachelor's degree or higher, 2019-23": "Bachelor's+ %",
}

print(f"Loaded {len(analysis_df)} counties | Features: {FEATURES}")

---
## 1. Baseline OLS — Linear Benchmark

OLS with HC3 robust standard errors. Both unemployment and poverty enter linearly. This establishes direction and magnitude but imposes a constant marginal effect across the full range of each predictor.

In [ ]:
X_ols = analysis_df[FEATURES]
X_ols = sm.add_constant(X_ols)
y_ols = analysis_df[OUTCOME]

ols_fit = sm.OLS(y_ols, X_ols)
ols_model = ols_fit.fit(cov_type='HC3')

print(ols_model.summary())

**Figure 3.** Baseline OLS coefficients with HC3 95% confidence intervals. Both poverty rate and unemployment rate carry positive, statistically significant coefficients.

In [ ]:
display(Image('figures/ols_coefficients.png', width=700))

r2 = round(ols_model.rsquared, 3)
adj_r2 = round(ols_model.rsquared_adj, 3)
n_obs = int(ols_model.nobs)
print(f"R\u00b2 = {r2}  |  Adj. R\u00b2 = {adj_r2}  |  N = {n_obs}")

---
## 2. Nonlinear OLS Specifications

Three OLS specs for unemployment. Improvement in R², AIC, or BIC from the linear to nonlinear spec is evidence against the constant-marginal-effect assumption.

In [ ]:
control_vars = []
for col in [POV] + CONTROLS:
    if col in analysis_df.columns:
        control_vars.append(f'Q("{col}")')
ctrl_f = ' + '.join(control_vars)

# Model A: linear unemployment
formula_A = f'Q("{OUTCOME}") ~ Q("{UNEM}") + {ctrl_f}'
mA = smf.ols(formula_A, data=analysis_df).fit(cov_type='HC3')

# Model B: quadratic unemployment
analysis_df['unem_sq'] = analysis_df[UNEM] ** 2
formula_B = f'Q("{OUTCOME}") ~ Q("{UNEM}") + unem_sq + {ctrl_f}'
mB = smf.ols(formula_B, data=analysis_df).fit(cov_type='HC3')

# Model C: unemployment quintiles
analysis_df['unem_q5'] = pd.qcut(analysis_df[UNEM], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
formula_C = f'Q("{OUTCOME}") ~ C(unem_q5) + {ctrl_f}'
mC = smf.ols(formula_C, data=analysis_df).fit(cov_type='HC3')

rows = []

row_A = {'Model': 'A: Linear', 'R2': round(mA.rsquared, 4), 'Adj R2': round(mA.rsquared_adj, 4), 'AIC': round(mA.aic, 1), 'BIC': round(mA.bic, 1)}
rows.append(row_A)

row_B = {'Model': 'B: Quadratic', 'R2': round(mB.rsquared, 4), 'Adj R2': round(mB.rsquared_adj, 4), 'AIC': round(mB.aic, 1), 'BIC': round(mB.bic, 1)}
rows.append(row_B)

row_C = {'Model': 'C: Quintiles', 'R2': round(mC.rsquared, 4), 'Adj R2': round(mC.rsquared_adj, 4), 'AIC': round(mC.aic, 1), 'BIC': round(mC.bic, 1)}
rows.append(row_C)

print(pd.DataFrame(rows).to_string(index=False))

---
## 3. Machine Learning Models

Linear Regression, Random Forest, and XGBoost trained on the same feature set (80/20 split, `random_state=42`). Tree-based models capture nonlinearities and interactions that OLS cannot.

In [ ]:
X_ml = analysis_df[FEATURES].copy()
y_ml = analysis_df[OUTCOME].copy()

X_train, X_test, y_train, y_test = train_test_split(X_ml, y_ml, test_size=0.2, random_state=42)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Random Forest
rf = RandomForestRegressor(n_estimators=500, max_depth=6, min_samples_leaf=10,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=3,
                               subsample=0.8, colsample_bytree=0.8, reg_lambda=5,
                               random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

# evaluate Linear Regression
lr_train_pred = lr.predict(X_train)
lr_test_pred = lr.predict(X_test)
lr_row = {
    'Model': 'Linear Regression',
    'Train R2': round(r2_score(y_train, lr_train_pred), 3),
    'Test R2': round(r2_score(y_test, lr_test_pred), 3),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, lr_test_pred)), 1),
    'MAE': round(mean_absolute_error(y_test, lr_test_pred), 1)
}

# evaluate Random Forest
rf_train_pred = rf.predict(X_train)
rf_test_pred = rf.predict(X_test)
rf_row = {
    'Model': 'Random Forest',
    'Train R2': round(r2_score(y_train, rf_train_pred), 3),
    'Test R2': round(r2_score(y_test, rf_test_pred), 3),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, rf_test_pred)), 1),
    'MAE': round(mean_absolute_error(y_test, rf_test_pred), 1)
}

# evaluate XGBoost
xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)
xgb_row = {
    'Model': 'XGBoost',
    'Train R2': round(r2_score(y_train, xgb_train_pred), 3),
    'Test R2': round(r2_score(y_test, xgb_test_pred), 3),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, xgb_test_pred)), 1),
    'MAE': round(mean_absolute_error(y_test, xgb_test_pred), 1)
}

results = pd.DataFrame([lr_row, rf_row, xgb_row])
print(results.to_string(index=False))
results.to_csv('output/model_comparison.csv', index=False)

**Figure 4.** Out-of-sample R² comparison across models. XGBoost achieves the highest test R², confirming that the crime–socioeconomics relationship contains structure beyond what linear regression can capture.

In [ ]:
display(Image('figures/model_comparison.png', width=700))

---
## 4. SHAP Explainability

SHAP values decompose each XGBoost prediction into feature-level contributions. The bar chart ranks predictors by mean absolute SHAP value — a model-agnostic importance measure.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_vals = explainer(X_test)

feat_labels = []
for f in FEATURES:
    if f in SHORT:
        feat_labels.append(SHORT[f])
    else:
        feat_labels.append(f)

shap_abs = np.abs(shap_vals.values)
mean_shap = shap_abs.mean(axis=0)

shap_df = pd.DataFrame({
    'Feature': feat_labels,
    'Mean |SHAP|': mean_shap.round(4)
})

shap_sorted = shap_df.sort_values('Mean |SHAP|', ascending=False)
print(shap_sorted.to_string(index=False))

**Figure 5.** Mean absolute SHAP values (XGBoost). Poverty rate ranks as the most important predictor, followed by median household income.

In [ ]:
display(Image('figures/shap_importance.png', width=700))

---
## 5. Threshold Analysis *(Primary Finding)*

Counties are grouped into narrow fixed-width bins (1 pp for poverty, 0.5 pp for unemployment). Mean observed and XGBoost-predicted crime are computed per bin. A threshold is reported only when the slope increase at a given bin exceeds 1.5× the average slope change across all bins.

In [ ]:
def make_bins(data, predictor, outcome, xgb_col, bin_width, min_n=40):
    df = data.copy()

    lo = np.floor(df[predictor].min() / bin_width) * bin_width
    hi = np.ceil(df[predictor].max() / bin_width) * bin_width + bin_width
    bins = np.arange(lo, hi, bin_width)

    df['_b'] = pd.cut(df[predictor], bins=bins)

    groups = df.groupby('_b', observed=True)
    mean_pred_vals = groups[predictor].mean()
    mean_obs_vals = groups[outcome].mean()
    mean_xgb_vals = groups[xgb_col].mean()
    count_vals = groups[outcome].count()

    stats = pd.DataFrame({
        'mean_pred': mean_pred_vals,
        'mean_obs': mean_obs_vals,
        'mean_xgb': mean_xgb_vals,
        'n': count_vals
    }).reset_index()

    stats = stats[stats['n'] >= min_n].dropna()

    if len(stats) < 5:
        df['_b'] = pd.qcut(df[predictor], q=10, duplicates='drop')
        groups = df.groupby('_b', observed=True)
        mean_pred_vals = groups[predictor].mean()
        mean_obs_vals = groups[outcome].mean()
        mean_xgb_vals = groups[xgb_col].mean()
        count_vals = groups[outcome].count()

        stats = pd.DataFrame({
            'mean_pred': mean_pred_vals,
            'mean_obs': mean_obs_vals,
            'mean_xgb': mean_xgb_vals,
            'n': count_vals
        }).reset_index()

    stats = stats.sort_values('mean_pred').reset_index(drop=True)
    return stats


def estimate_threshold(stats, ratio=1.5):
    if len(stats) < 4:
        return None

    xvals = stats['mean_pred'].values
    yvals = stats['mean_xgb'].values

    slopes = []
    for i in range(len(yvals) - 1):
        rise = yvals[i+1] - yvals[i]
        run = xvals[i+1] - xvals[i]
        slopes.append(rise / run)

    slopes = np.array(slopes)

    if len(slopes) < 3:
        return None

    changes = []
    for i in range(len(slopes) - 1):
        changes.append(slopes[i+1] - slopes[i])

    changes = np.array(changes)
    idx = int(np.argmax(changes))
    avg = np.mean(np.abs(changes))

    if avg > 0 and changes[idx] > ratio * avg:
        return xvals[idx + 1]
    else:
        return None


analysis_df['xgb_pred'] = xgb_model.predict(analysis_df[FEATURES])

pov_stats = make_bins(analysis_df, POV, OUTCOME, 'xgb_pred', bin_width=1.0)
unem_stats = make_bins(analysis_df, UNEM, OUTCOME, 'xgb_pred', bin_width=0.5)

pov_thresh = estimate_threshold(pov_stats)
unem_thresh = estimate_threshold(unem_stats)

if pov_thresh is not None:
    print("Poverty threshold:    ", str(round(pov_thresh, 1)) + "%")
else:
    print("Poverty threshold:     not detected")

if unem_thresh is not None:
    print("Unemployment threshold:", str(round(unem_thresh, 1)) + "%")
else:
    print("Unemployment threshold: not detected")

### Figure 6 — Poverty Threshold Curve *(Most Important)*

The vertical line marks the estimated poverty breakpoint where XGBoost-predicted crime begins to accelerate. Bars show mean observed crime per bin; the dashed line shows XGBoost predictions.

In [ ]:
display(Image('figures/poverty_threshold_curve.png', width=760))

if pov_thresh is not None:
    above_rows = pov_stats[pov_stats['mean_pred'] > pov_thresh]
    below_rows = pov_stats[pov_stats['mean_pred'] <= pov_thresh]
    above = above_rows['mean_xgb'].mean()
    below = below_rows['mean_xgb'].mean()
    pct = 100 * (above - below) / max(abs(below), 1)
    print(f"Estimated poverty threshold: {round(pov_thresh, 1)}%")
    print(f"Predicted crime above: {round(above)}  |  below: {round(below)}  |  difference: +{round(pct)}%")
else:
    print("No clear poverty threshold detected.")

### Figure 7 — Unemployment Threshold Curve

Unlike poverty, the unemployment–crime relationship is more gradual. This curve tests whether a comparable tipping point exists for unemployment.

In [ ]:
display(Image('figures/unemployment_threshold_curve.png', width=760))

if unem_thresh is not None:
    above_rows = unem_stats[unem_stats['mean_pred'] > unem_thresh]
    below_rows = unem_stats[unem_stats['mean_pred'] <= unem_thresh]
    above = above_rows['mean_xgb'].mean()
    below = below_rows['mean_xgb'].mean()
    pct = 100 * (above - below) / max(abs(below), 1)
    print(f"Estimated unemployment threshold: {round(unem_thresh, 1)}%")
    print(f"Predicted crime above: {round(above)}  |  below: {round(below)}  |  difference: +{round(pct)}%")
else:
    print("No clear unemployment threshold detected (gradual, monotonic increase).")

In [ ]:
if pov_thresh is None:
    pov_interp = 'Gradual increase; no clear breakpoint.'
    pov_thresh_str = 'None detected'
else:
    above_pov = pov_stats[pov_stats['mean_pred'] > pov_thresh]['mean_xgb'].mean()
    below_pov = pov_stats[pov_stats['mean_pred'] <= pov_thresh]['mean_xgb'].mean()
    pct_pov = 100 * (above_pov - below_pov) / max(abs(below_pov), 1)
    pov_interp = f"Predicted crime ~{round(pct_pov)}% higher above {round(pov_thresh, 1)}% than below."
    pov_thresh_str = str(round(pov_thresh, 1)) + "%"

if unem_thresh is None:
    unem_interp = 'Gradual increase; no clear breakpoint.'
    unem_thresh_str = 'None detected'
else:
    above_unem = unem_stats[unem_stats['mean_pred'] > unem_thresh]['mean_xgb'].mean()
    below_unem = unem_stats[unem_stats['mean_pred'] <= unem_thresh]['mean_xgb'].mean()
    pct_unem = 100 * (above_unem - below_unem) / max(abs(below_unem), 1)
    unem_interp = f"Predicted crime ~{round(pct_unem)}% higher above {round(unem_thresh, 1)}% than below."
    unem_thresh_str = str(round(unem_thresh, 1)) + "%"

summary = pd.DataFrame([
    {
        'Variable': 'Poverty Rate (PCTPOVALL_2023)',
        'Threshold': pov_thresh_str,
        'Interpretation': pov_interp
    },
    {
        'Variable': 'Unemployment Rate (Unemployment_rate_2023)',
        'Threshold': unem_thresh_str,
        'Interpretation': unem_interp
    }
])

pd.set_option('display.max_colwidth', 120)
print(summary.to_string(index=False))
summary.to_csv('output/threshold_summary.csv', index=False)

---
## 6. Summary of Findings

1. **Poverty is the dominant predictor.** SHAP importance ranks poverty rate above unemployment, capturing more county-level crime variation once other controls are held constant.

2. **Unemployment has a positive linear association with possible acceleration at higher levels.** The OLS coefficient is positive and significant; the threshold curve characterizes whether the slope steepens at the upper end.

3. **Nonlinear OLS improves fit.** The quadratic and quintile-bin models achieve lower AIC/BIC than the strictly linear baseline, confirming that the unemployment–crime relationship is not constant-marginal.

4. **XGBoost outperforms OLS out-of-sample.** The test-set R² gain reflects the model capturing interactions and nonlinearities that OLS cannot.

5. **OLS systematically undershoots the most distressed counties.** Absolute error rises in the highest unemployment and poverty deciles — the signature of a missed nonlinear effect.

### Conclusion

Socioeconomic distress predicts violent crime, but the relationship is not equally strong across variables. **Poverty is the dominant predictor** in the XGBoost model; **unemployment shows a clearer nonlinear pattern** at moderate-to-high levels. Threshold estimates (Section 5) are data-driven and should be read descriptively — this analysis is predictive and correlational, not causal.